# Hypothesis 1 — Main Notebook

This notebook generates 33 experiment configurations, runs them in parallel with configurable concurrency and selection, logs outputs, and analyzes how often groups reach each of the four principles vs. disagreement.

In [1]:
# Imports
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'hypothesis_testing').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
from collections import Counter
import numpy as np
from scipy.stats import chi2_contingency
from hypothesis_testing.utils_hypothesis_testing.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


In [ ]:
# Configuration paths and constants
CONFIG_DIR = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_1' / 'configs'
TERMINAL_OUTPUTS_DIR   = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_1' / 'terminal_outputs'
RESULTS_DIR= _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_1' / 'results'
TRANSCRIPTS_DIR = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_1' / 'transcripts'

# Placeholder model list for participant agents

MODEL_LIST = [
    "gemini-2.5-pro",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

# Income class probabilities (must sum to 1.0)
# Same as in Frohlich & Oppenheimer (1992) for the initial distribution
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# Ensure directories exist
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TERMINAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_DIR, TERMINAL_OUTPUTS_DIR, RESULTS_DIR, TRANSCRIPTS_DIR


(PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_1/configs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_1/terminal_outputs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_1/results'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_1/transcripts'))

## 1. Generate 33 Configurations

In [4]:
def make_agents(temp: float, rng: random.Random) -> list[dict]:
    agents = []
    for i in range(0, 5):  # 5 participant agents
        agents.append({
            'name': f'Agent_{i+1}',
            'personality': 'You are an American college student',
            'model': rng.choice(MODEL_LIST),
            'temperature': float(temp),
            'memory_character_limit': 25000,
            'reasoning_enabled': True,
        })
    return agents

def build_config(temp: float, seed_val: int, rng: random.Random) -> dict:
    return {
        'language': 'English',
        'seed': int(seed_val),
        'agents': make_agents(temp, rng),
        'utility_agent_model': 'gemini-2.0-flash-lite-001',
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 10,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': {'enabled': True},
    }

# Temperatures: 11 with 0, 11 with U(0,1), 11 with U(0,1.5)
GLOBAL_SEED = 21000
master_rng = random.Random(GLOBAL_SEED)  # deterministic config generation

temps_fixed = [0.0] * 11
temps_u01 = [master_rng.uniform(0.0, 1.0) for _ in range(11)]
temps_u015 = [master_rng.uniform(0.0, 1.5) for _ in range(11)]
all_temps = temps_fixed + temps_u01 + temps_u015

generated_files = []
for idx, temp in enumerate(all_temps, start=1):
    seed_val = master_rng.randint(0, 2**31 - 1)
    cfg_rng = random.Random(seed_val)  # tie agent sampling to the seed
    cfg = build_config(temp=temp, seed_val=seed_val, rng=cfg_rng)
    fname = CONFIG_DIR / f'hypothesis_1_condition_{idx}_config.yaml'
    with open(fname, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    generated_files.append(fname)

len(generated_files), generated_files[0] if generated_files else None


(33,
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_1/configs/hypothesis_1_condition_1_config.yaml'))

## 2. Run Configs (Parallel + Selective)

In [ ]:
# Discover all config files
configs = list_config_files(CONFIG_DIR)
print(f'Found {len(configs)} configs')

# Selection controls
SELECT_INDICES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 
11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 
21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
31, 32, 33]  # e.g., [1,2,3] for the first three
SELECT_NAMES = None    # e.g., ['condition_1', 'condition_33']
CONCURRENCY = 5        # adjust parallel workers
TIMEOUT_SECONDS = None # e.g., 900 for 15 minutes per run

selected = select_configs(configs, include_indices=SELECT_INDICES, include_names=SELECT_NAMES)
print(f'Selected {len(selected)} configs to run')

run_results = run_configs_in_parallel(
    selected,
    concurrency=CONCURRENCY,
    logs_dir=TERMINAL_OUTPUTS_DIR,
    results_dir=RESULTS_DIR,
    timeout_sec=TIMEOUT_SECONDS,
)

# Quick summary
ok = sum(1 for r in run_results if r.get('ok'))
print(f'Completed: {ok}/{len(run_results)} OK')
run_results[:3]  # show a few

Found 33 configs
Selected 33 configs to run


## 3. Analysis — Principles vs. Disagreements

Counts how often runs ended in consensus for each principle vs. disagreement (no consensus).

In [10]:

CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

counts = Counter()
result_files = sorted(RESULTS_DIR.glob('*_results.json'))
for rp in result_files:
    counts[categorize_result(rp)] += 1

# Ensure all categories are present
for cat in CATEGORIES:
    counts.setdefault(cat, 0)

# Display as a simple table
print('Category | Count')
print('---------|------')
for cat in CATEGORIES:
    print(f'{cat:38} | {counts[cat]}')

counts

Category | Count
---------|------
maximizing_floor                       | 2
maximizing_average                     | 0
maximizing_average_floor_constraint    | 8
maximizing_average_range_constraint    | 0
disagreement                           | 2


Counter({'maximizing_average_floor_constraint': 8,
         'disagreement': 2,
         'maximizing_floor': 2,
         'maximizing_average': 0,
         'maximizing_average_range_constraint': 0})

## 4. Statistical Tests — Fisher–Freeman–Halton and Cramér's V

Compare aggregated Hypothesis 1 outcomes (AI) against human outcomes as a 5×2 contingency table.

- Rows (categories): the four principles + disagreement.
- Columns (groups): Human vs AI.
- Fisher–Freeman–Halton exact test via R's `fisher.test()` when available; fallback to Chi-square otherwise.
- Cramér's V with bias correction and bootstrap CI.


In [11]:

# 1) Aggregate AI outcomes across all runs into 5 categories
ai_counts = [counts.get(cat, 0) for cat in CATEGORIES]
print('AI counts by category:', dict(zip(CATEGORIES, ai_counts)))

# 2) Specify Human counts (edit to match hypothesis_testing/hypothesis_1/image copy.png)
# Defaults below use Frohlich & Oppenheimer published values as a placeholder.
HUMAN_COUNTS = {
    'maximizing_floor': 5,
    'maximizing_average': 1,
    'maximizing_average_floor_constraint': 23,
    'maximizing_average_range_constraint': 2,
    'disagreement': 7,
}
human_counts = [HUMAN_COUNTS.get(cat, 0) for cat in CATEGORIES]
print('Human counts by category:', dict(zip(CATEGORIES, human_counts)))

# 3) Build 5×2 contingency table (rows=categories, cols=[Human, AI])
contingency = np.vstack([human_counts, ai_counts]).T  # shape (5, 2)
contingency, CATEGORIES, ['Human','AI']


AI counts by category: {'maximizing_floor': 2, 'maximizing_average': 0, 'maximizing_average_floor_constraint': 8, 'maximizing_average_range_constraint': 0, 'disagreement': 2}
Human counts by category: {'maximizing_floor': 5, 'maximizing_average': 1, 'maximizing_average_floor_constraint': 23, 'maximizing_average_range_constraint': 2, 'disagreement': 7}


(array([[ 5,  2],
        [ 1,  0],
        [23,  8],
        [ 2,  0],
        [ 7,  2]]),
 ['maximizing_floor',
  'maximizing_average',
  'maximizing_average_floor_constraint',
  'maximizing_average_range_constraint',
  'disagreement'],
 ['Human', 'AI'])

In [14]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    """Run Fisher-Freeman-Halton test via R's fisher.test if available.
    Returns p-value or None if Rscript not found or fails.
    """
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow = contingency.shape[0]
    r_code = f"""
m <- matrix(c({r_matrix}), nrow={nrow}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) {{ cat(f$p.value) }} else {{ cat('NA') }}
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except Exception:
        return None

def chi2_fallback_pvalue(contingency: np.ndarray) -> tuple[float, float, int, np.ndarray]:
    chi2, p, dof, expected = chi2_contingency(contingency)
    return chi2, p, dof, expected

p_ffh = fisher_freeman_halton_pvalue_r(contingency)
if p_ffh is None:
    chi2, p_chi, dof, exp = chi2_fallback_pvalue(contingency)
    print('R not available or exact test failed — using Chi-square approximation')
    print(f'Chi-square test: chi2={chi2:.4f}, dof={dof}, p={p_chi:.6f}')
else:
    print(f'Fisher–Freeman–Halton exact test p-value: {p_ffh:.6f}')


Fisher–Freeman–Halton exact test p-value: 1.000000


In [13]:
def cramers_v(contingency: np.ndarray) -> float:
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, c - 1))))

def bias_corrected_cramers_v(contingency: np.ndarray) -> float:
    # Bergsma (2013) bias correction
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    phi2 = chi2 / n
    r1 = r - 1
    c1 = c - 1
    phi2_corr = max(0.0, phi2 - (r1 * c1) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    denom = min(r_corr - 1, c_corr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))

def bootstrap_cramers_v(contingency: np.ndarray, n_bootstrap: int = 10000, confidence_level: float = 0.95, bias_corrected: bool = True, seed: int | None = 123) -> tuple[np.ndarray, float, float]:
    rng = np.random.default_rng(seed)
    n = int(contingency.sum())
    r, c = contingency.shape
    # Expand to individual pairs
    pairs = []
    for i in range(r):
        for j in range(c):
            pairs.extend([(i, j)] * int(contingency[i, j]))
    pairs = np.array(pairs)
    if len(pairs) == 0:
        return np.array([]), 0.0, 0.0
    vs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(pairs), size=n)
        sample = pairs[idx]
        # Re-tabulate
        tab = np.zeros_like(contingency)
        for (i, j) in sample:
            tab[i, j] += 1
        v = bias_corrected_cramers_v(tab) if bias_corrected else cramers_v(tab)
        vs.append(v)
    vs = np.array(vs)
    alpha = 1 - confidence_level
    lo, hi = np.quantile(vs, [alpha / 2, 1 - alpha / 2])
    return vs, float(lo), float(hi)

v_std = cramers_v(contingency) if contingency.sum() > 0 else 0.0
v_corr = bias_corrected_cramers_v(contingency) if contingency.sum() > 0 else 0.0
print(f"Cramér's V (standard): {v_std:.4f}")
print(f"Cramér's V (bias-corrected): {v_corr:.4f}")

# Bootstrap CI (adjust iterations if needed for speed)
if contingency.sum() > 0:
    boot_vs, ci_lo, ci_hi = bootstrap_cramers_v(contingency, n_bootstrap=5000, confidence_level=0.95, bias_corrected=True, seed=123)
    print(f"95% CI for bias-corrected V: [{ci_lo:.4f}, {ci_hi:.4f}]")
else:
    print('Insufficient data for bootstrap CI')


Cramér's V (standard): 0.1482
Cramér's V (bias-corrected): 0.0000


ValueError: The internally computed table of expected frequencies has a zero element at (np.int64(1), np.int64(0)).